# Khmer Legal Semantic Search

Search ChromaDB chunks and return complete parent legal articles with matched-chunk evidence.

In [1]:
import json
import logging
import math
import unicodedata
from pathlib import Path

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.set_verbosity_error()
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

def find_project_root(start: Path) -> Path:
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Run this notebook inside the legal-rag project')

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
ARTICLE_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
CHROMA_DIR = DATA_DIR / 'processed' / 'chroma_db'
TOP_K = 5
MODELS = {
    'sealion': {'collection': 'legal_chunks_sealion', 'model': 'aisingapore/SEA-LION-E5-Embedding-600M'},
    'granite-r2': {'collection': 'legal_chunks_granite_r2', 'model': 'ibm-granite/granite-embedding-311m-multilingual-r2'},
    'bge-m3': {'collection': 'legal_chunks_bge_m3', 'model': 'BAAI/bge-m3'},
}

if not ARTICLE_PATH.exists():
    raise FileNotFoundError(f'Missing normalized articles: {ARTICLE_PATH}. Run normalize_articles.ipynb first.')
ARTICLES_BY_ID = {}
for line in ARTICLE_PATH.read_text(encoding='utf-8').splitlines():
    if line.strip():
        article = json.loads(line)
        ARTICLES_BY_ID[str(article['id'])] = article

CHROMA = chromadb.PersistentClient(path=str(CHROMA_DIR))
COLLECTION_NAMES = {collection.name for collection in CHROMA.list_collections()}
AVAILABLE_MODELS = [key for key, config in MODELS.items() if config['collection'] in COLLECTION_NAMES]
if not AVAILABLE_MODELS:
    raise RuntimeError('No ChromaDB collections found. Run store_chunk_embeddings.ipynb first.')
print('Available models:', AVAILABLE_MODELS)

W1101 00:15:26.299000 57027 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Available models: ['sealion', 'granite-r2', 'bge-m3']


In [2]:
_QUERY_MODELS = {}

def normalize_query(query: str) -> str:
    query = unicodedata.normalize('NFC', str(query))
    for character in ('\u200b', '\u200c', '\u200d', '\ufeff'):
        query = query.replace(character, '')
    return ' '.join(query.replace('\u00a0', ' ').split())

def embed_query(query: str, model_key: str) -> np.ndarray:
    model = _QUERY_MODELS.get(model_key)
    if model is None:
        model = SentenceTransformer(MODELS[model_key]['model'])
        _QUERY_MODELS[model_key] = model
    vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
    return np.asarray(vector, dtype='float32')

def get_parent_article(metadata: dict, chunk_id: str) -> dict | None:
    parent_id = str(metadata.get('parent_id') or chunk_id.rsplit('-chunk-', 1)[0])
    article = ARTICLES_BY_ID.get(parent_id)
    if article is not None:
        return article
    document_id = str(metadata.get('document_id', ''))
    article_number = metadata.get('article_number')
    return next((item for item in ARTICLES_BY_ID.values() if str(item.get('document_id')) == document_id and item.get('article_number') == article_number), None)

def search_articles(query: str, model_key: str, top_k: int = TOP_K) -> list[dict]:
    query = normalize_query(query)
    if not query:
        raise ValueError('Query cannot be empty')
    if model_key not in AVAILABLE_MODELS:
        raise ValueError(f'Model {model_key!r} is not available. Available: {AVAILABLE_MODELS}')
    collection = CHROMA.get_collection(MODELS[model_key]['collection'])
    candidate_count = min(collection.count(), max(top_k * 5, 20))
    result = collection.query(query_embeddings=[embed_query(query, model_key).tolist()], n_results=candidate_count, include=['documents', 'metadatas', 'distances'])

    parents = {}
    for index, chunk_id in enumerate(result['ids'][0]):
        metadata = result['metadatas'][0][index] or {}
        article = get_parent_article(metadata, chunk_id)
        if article is None:
            continue
        parent_id = str(article['id'])
        match = {'chunk_id': chunk_id, 'distance': float(result['distances'][0][index]), 'text': result['documents'][0][index]}
        parents.setdefault(parent_id, {'article': article, 'matches': []})['matches'].append(match)

    ranked = sorted(parents.values(), key=lambda item: min(match['distance'] for match in item['matches']))
    output = []
    for rank, item in enumerate(ranked[:top_k], start=1):
        article = item['article']
        matches = sorted(item['matches'], key=lambda match: match['distance'])
        output.append({'rank': rank, 'article_id': article.get('article_id'), 'document_id': article.get('document_id'), 'article_number': article.get('article_number'), 'distance': matches[0]['distance'], 'matched_chunks': matches[:2], 'text': article.get('text', ''), 'title': article.get('document_title'), 'source_url': article.get('source_url'), 'pdf_url': article.get('pdf_url')})
    return output

In [3]:
query = 'តើការបើកបរដោយប្រើគ្រឿងស្រវឹងខុសឬទេ?'
all_results = {model_key: search_articles(query, model_key) for model_key in AVAILABLE_MODELS}
for model_key, results in all_results.items():
    print(f'\n=== {model_key} ===')
    for result in results:
        print(f"{result['rank']}. {result['document_id']}-{result['article_number']} | distance={result['distance']:.4f} | {result['title']}")

# Full article text remains available in all_results.
# Example: all_results['bge-m3'][0]['text']


=== sealion ===
1. 1013-70 | distance=0.2818 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់
2. 1013-47 | distance=0.2820 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់
3. 1004-135 | distance=0.2842 | ច្បាប់ស្តីពីប្រកាសនីយបត្រតក្កកម្មវិញ្ញាបនបត្រម៉ូដែល មានអត្ថប្រយោជន៍ និងគំនូរឧស្សាហកម្ម
4. 1013-25 | distance=0.2844 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់
5. 1013-67 | distance=0.2844 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់

=== granite-r2 ===
1. 1013-25 | distance=0.1529 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់
2. 11169-10 | distance=0.1579 | អនុសញ្ញាក្រុងប៊ែន ស្ដីពីកិច្ចការពារស្នាដៃអក្សរសាស្ត្រ និងសិល្បៈ
3. 1013-47 | distance=0.1597 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់
4. 1013-26 | distance=0.1606 | ច្បាប់ស្តីពីម៉ាក ពាណិជ្ជនាម និងអំពើ នៃការប្រកួតប្រជែងមិនស្មោះត្រង់
5. 11173-5 | distance=0.1606 | អនុសាសន៍រួម ស្តីពីអាជ្ញាប័ណ្ណម៉

## Evaluation

Evaluation runs only for ChromaDB collections that are available.

In [4]:
EVALUATION_QUERIES = [
    ('តើច្បាប់នេះការពារអ្វីខ្លះទាក់ទងនឹងប៉ាតង់ និងគំនូរឧស្សាហកម្ម?', {('1004', 1)}),
    ('តើលក្ខខណ្ឌអ្វីខ្លះដែលធ្វើឱ្យការបង្កើតមួយអាចទទួលបានប៉ាតង់?', {('1004', 5)}),
    ('តើម៉ាកមានន័យដូចម្តេច ហើយវាប្រើសម្គាល់អ្វី?', {('1013', 2)}),
    ('តើការចុះបញ្ជីម៉ាកមានសុពលភាពរយៈពេលប៉ុន្មានឆ្នាំ?', {('1013', 12)}),
    ('តើស្នាដៃប្រភេទណាខ្លះត្រូវបានការពារតាមច្បាប់សិទ្ធិអ្នកនិពន្ធ?', {('1006', 3)}),
    ('តើការិយាល័យអន្តរជាតិមានតួនាទីអ្វីខ្លះក្នុងការការពារសិទ្ធិអ្នកនិពន្ធ?', {('11169', 24)}),
    ('តើកិច្ចព្រមព្រៀងនេះកំណត់ការចែកចាយសញ្ញាកម្មវិធីតាមផ្កាយរណបយ៉ាងដូចម្តេច?', {('11170', 2)}),
    ('តើអាជ្ញាប័ណ្ណម៉ាកត្រូវបង្ហាញឬទេ ហើយការមិនបង្ហាញប៉ះពាល់អ្វី?', {('11173', 6)}),
    ('តើកិច្ចព្រមព្រៀងក្រុងឡូការណូបង្កើតចំណាត់ថ្នាក់អ្វី?', {('11167', 1)}),
    ('តើគំនូរឧស្សាហកម្មដែលស្ថិតក្នុងក្រុមចំណាត់ថ្នាក់ដូចគ្នាអាចដាក់ស្នើរួមគ្នាបានទេ?', {('1004', 87)}),
]

def result_key(result):
    return str(result['document_id']), int(result['article_number'])

def score_results(results, relevant, k):
    found = [result_key(result) for result in results[:k]]
    positions = [index + 1 for index, item in enumerate(found) if item in relevant]
    first = positions[0] if positions else None
    return {f'hit@{k}': int(first is not None), f'recall@{k}': len(set(found) & relevant) / len(relevant), f'mrr@{k}': 1 / first if first else 0.0, f'ndcg@{k}': 1 / math.log2(first + 1) if first else 0.0}

report = {'models': {}}
for model_key in AVAILABLE_MODELS:
    rows = []
    for number, (query, relevant) in enumerate(EVALUATION_QUERIES, start=1):
        results = search_articles(query, model_key)
        scores = {**score_results(results, relevant, 3), **score_results(results, relevant, 5)}
        rows.append({'query_number': number, 'query': query, 'expected': sorted(f'{doc}-{article}' for doc, article in relevant), 'top5': [f"{item['document_id']}-{item['article_number']}" for item in results], **scores})
    names = ('hit@3', 'recall@3', 'mrr@3', 'ndcg@3', 'hit@5', 'recall@5', 'mrr@5', 'ndcg@5')
    report['models'][model_key] = {'collection': MODELS[model_key]['collection'], 'results': rows, 'metrics': {name: round(sum(row[name] for row in rows) / len(rows), 4) for name in names}}
    print(model_key, report['models'][model_key]['metrics'])

report_path = DATA_DIR / 'processed' / 'reports' / 'chunk_search_evaluation.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Saved:', report_path)

sealion {'hit@3': 0.8, 'recall@3': 0.8, 'mrr@3': 0.6667, 'ndcg@3': 0.7, 'hit@5': 0.8, 'recall@5': 0.8, 'mrr@5': 0.6667, 'ndcg@5': 0.7}
granite-r2 {'hit@3': 0.7, 'recall@3': 0.7, 'mrr@3': 0.5833, 'ndcg@3': 0.6131, 'hit@5': 0.8, 'recall@5': 0.8, 'mrr@5': 0.6033, 'ndcg@5': 0.6518}


bge-m3 {'hit@3': 0.9, 'recall@3': 0.9, 'mrr@3': 0.7833, 'ndcg@3': 0.8131, 'hit@5': 0.9, 'recall@5': 0.9, 'mrr@5': 0.7833, 'ndcg@5': 0.8131}
Saved: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/reports/chunk_search_evaluation.json
